# KAT-VAD v2.2 — **batch 2 / SC-1: DF8 position swap** (no training)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§24 (Amendment 13, X1–X10)** — it must be in the upload, and
should be **committed before this notebook runs**. Plan: `.project/plans/katvad-v2-df8-position-shift.md`.
Question: *does the T2 position prior `p_T2` still beat A3-ign when the accident is moved inside its clip?*

| step | what | GPU? |
|---|---|---|
| 0 | setup; refuses an upload without §24 / `position_shift` | — |
| 1 | preflight + stage (DoTA s1, V2-S DoTA-CAP, A3 T2 stats, saved protocol-B / prior scores) | no |
| 2 | `plan` — crop table from the frozen labels (must be **491** clips) | no |
| 3 | `heads` per CAP group — re-extract the V2-S rows whose 1.5 s window reaches before a crop start (D2), gate X10(c) | **yes** |
| 4 | `score` A3-ign (primary), A3, A0 × 5 seeds on full/control/early/late — gate X10(b): `full` == saved scores | **yes** |
| 5 | `prior` — `p_T2` per crop; gate X10(b) vs E3's prior | no |
| 6 | `readout` — R1–R4, X7, the X8 sentence | no |

DoTA-CAP-eval, DoTA-eval and T2-test are never read. Every step resumes (re-run the cell after a disconnect).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), data/, cache/v2, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V22_DF8_EXPECTED_CLIPS'), 'this upload predates DF8 -- re-upload branch v2'
assert (REPO / 'core/tools/position_shift.py').is_file(), 'upload has no core/tools/position_shift.py'
ADDENDUM = REPO / 'core/docs/v2/PREREG_ADDENDUM.md'
assert '## 24. Amendment 13' in ADDENDUM.read_text(encoding='utf-8'), 'upload has no addendum §24 (DF8 rule)'

DATASET  = constants.DADA_ORIGIN_DATASET                  # 'DADA2000_orig'
ENC      = constants.VIDEOMAE_ENCODER_S                   # A3 / A3-ign video stream
CRN      = 'R2'
SEEDS    = list(constants.V2_E3_SEEDS)                    # 2024-2028
CAP      = constants.V2_SPLIT_DOTA_CAP_DEV
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']         # CAP-DATA tar groups (dota_cap_videomae.ipynb)
BATCH_GB = constants.MMAU_STREAM_BATCH_GB

DOTA     = constants.DATA_ROOT / 'DoTA'
LABELS   = DOTA / 'labels_s8'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
ALIGN    = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap' / 'dota_cap_alignment.json'
V2_DATA  = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'     # T2-train - T2-val windows (p_T2's fit)
STATS    = Path(DRIVE_V2) / 'cache' / 'v2' / f'A3_{CRN}_S' / DATASET   # A3's T2 fit (= A3-ign's input)
E3       = constants.OUTPUT_ROOT / 'v2_e3'
E40      = constants.OUTPUT_ROOT / 'v2_e4_0'
OUT      = constants.OUTPUT_ROOT / 'REPORTS' / 'v22_df8'
LOCAL    = Path('/content/df8')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 3-4 need one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

ENV = dict(os.environ)


def run(cmd, log=None):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    # Copy a Drive dir (only ``ids`` + its manifests when given) to VM disk once, bit-identical (C2).
    t0 = time.time()
    if ids is None:
        if not dst.exists():
            shutil.copytree(src, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst

## 1. Preflight + stage

In [ ]:
from core.data.v2_inputs import STATS_FILENAME
from core.data.v2_splits import load_split
from core.tools import dota_cap, e2d_probe, e3_readout, position_prior, position_shift, protocol_b_eval

cap_dev = load_split(CAP)                                  # sha1-checked; eval never loaded
assert len(cap_dev) == 569, len(cap_dev)
S_DOTA = e2d_probe.dota_video_dir(constants.VIDEO_CACHE_DIR, ENC)
e2d_probe.check_manifest(S_DOTA, dota_cap.video_manifest(ENC, e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)))

CKPT = {'A3ign': lambda s: E40 / 'A3ign' / f's{s}' / 'checkpoint_last.pt',
        'A3':    lambda s: E3 / 'A3' / f's{s}' / 'checkpoint_last.pt',
        'A0':    lambda s: E3 / 'A0' / f's{s}' / 'checkpoint_last.pt'}
REF = {'A3ign': e3_readout.pb_dir(E40, 'A3ign', CAP) / protocol_b_eval.CLIP_SCORES_NPZ,
       'A3':    e3_readout.pb_dir(E3, 'A3', CAP) / protocol_b_eval.CLIP_SCORES_NPZ,
       'A0':    e3_readout.pb_dir(E3, 'A0', CAP) / protocol_b_eval.CLIP_SCORES_NPZ}
PRIOR_REF = E3 / e3_readout.REPORTS / e3_readout.PRIOR_DIR / CAP / position_prior.PRIOR_NPZ
need = [ALIGN, V2_DATA, PRIOR_REF, STATS / STATS_FILENAME, STATS / constants.V2_INPUT_MANIFEST_FILENAME,
        LABELS / constants.DEFS_FILENAME, RAW / 'cap_text_annotations.xls',
        *REF.values(), *(CKPT[m](s) for m in CKPT for s in SEEDS)]
miss = [str(p) for p in need if not p.exists()]
assert not miss, f'missing on Drive: {miss[:5]}'
for d, what in ((DOTA_S1, 'CLIP DoTA_s1_ncc'), (S_DOTA, 'V2-S DoTA-CAP')):
    m = [v for v in cap_dev if not (d / f'{v}.npy').exists()]
    assert not m, f'{what}: {len(m)} missing, e.g. {m[:3]}'
print('preflight OK')

st = LOCAL / 'stage'
L_S1 = stage(DOTA_S1, st / 'DoTA_s1_ncc', cap_dev)
L_U = stage(S_DOTA, st / 'v2s_cap', cap_dev)
L_STATS = st / 'stats'
L_STATS.mkdir(parents=True, exist_ok=True)
for name in (STATS_FILENAME, constants.V2_INPUT_MANIFEST_FILENAME):
    shutil.copy2(STATS / name, L_STATS / name)
L_LABELS = st / 'labels'
L_LABELS.mkdir(parents=True, exist_ok=True)
shutil.copy2(LABELS / constants.DEFS_FILENAME, L_LABELS / constants.DEFS_FILENAME)
L_V2 = stage(V2_DATA, st / 'corpus_v2')

## 2. Plan — the crop table (X2: must be 491 clips)

In [ ]:
CROPS = OUT / position_shift.CROPS_JSON
if not CROPS.exists():
    run([sys.executable, '-m', 'core.tools.position_shift', 'plan', '--s1-dir', L_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt', '--out-dir', OUT])
print((OUT / position_shift.CROPS_MD).read_text())

## 3. Heads — re-extract the clamped V2-S windows (D2), per CAP group — GPU

Same streaming as `dota_cap_videomae.ipynb`: only the CAP videos of the 491 clips are written, the tar stops once the
last one is closed. ≈ 5 windows per shifted arm + 1 gate window (X10(c): an unclamped window must equal the cache
row, cos ≥ 0.999). Re-running skips clips that already have `heads/{clip}.npz`.

In [ ]:
import pandas as pd

HEADS = OUT / 'heads'
crops = json.loads(CROPS.read_text())['clips']
alignment = json.loads(ALIGN.read_text())
ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
cap_of = {v: alignment['clips'][v]['cap_id'] for v in crops}
assert set(cap_of.values()) <= type_of.keys()
free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f} -- lower BATCH_GB'

for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    todo = sorted({cap_of[v] for v in crops if lo <= type_of[cap_of[v]] <= hi
                   and not (HEADS / f'{v}.npz').exists()})
    print(f'group {g}: {len(todo)} CAP videos to stream')
    if not todo:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in todo))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.position_shift', 'heads', '--crops', CROPS,
         '--alignment', ALIGN, '--parts', *parts, '--cap-ids-file', ids_file, '--video-dir', L_U,
         '--encoder', ENC, '--work-dir', LOCAL / 'work' / g, '--batch-size', '32',
         '--max-batch-gb', str(BATCH_GB), '--device', 'auto', '--out-dir', HEADS], log=OUT / 'heads_run.log')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

report = json.loads((HEADS / position_shift.HEADS_REPORT).read_text())
bad = {v: e for v, e in report.items() if e['reason'] != 'ok'}
done = [v for v in crops if (HEADS / f'{v}.npz').exists()]
cos = [e['gate_min_cos'] for e in report.values() if e.get('gate_min_cos') is not None]
print(f'heads: {len(done)} / {len(crops)} clips; not ok: {len(bad)}; gate cos min {min(cos):.6f}')
assert not bad, f'X10(c) / stream failures -- paste to Claude: {dict(list(bad.items())[:5])}'
assert len(done) == len(crops), 'some clips have no heads -- re-run this cell'

## 4. Score A3-ign (primary), A3, A0 — GPU

All 5 seeds per model in one call, seed-averaged per frame. Gate X10(b): each model's `full` arm must reproduce its
saved protocol-B scores on DoTA-CAP-dev (E4-0 / E3) within 1e-4 — otherwise the tool stops.

In [ ]:
for model in ('A3ign', 'A3', 'A0'):
    if (OUT / position_shift.scores_json(model)).exists():
        print('reusing', model)
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', CKPT[model](s))]
    extra = [] if model == 'A0' else ['--video-dir', L_U, '--stats-dir', L_STATS, '--heads-dir', HEADS]
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.position_shift', 'score', '--crops', CROPS, '--model', model,
         *runs, '--s1-dir', L_S1, *extra, '--data-dir', L_LABELS, '--reference', REF[model],
         '--device', 'auto', '--out-dir', OUT], log=OUT / f'score_{model}.log')
    print(f'{model}: {(time.time() - t0) / 60:.1f} min')
for model in ('A3ign', 'A3', 'A0'):
    meta = json.loads((OUT / position_shift.scores_json(model)).read_text())
    print(model, 'steps', meta['global_steps'], '| X10(b) max |diff|', f"{meta['x10b_max_abs_diff']:.1e}")

## 5. `p_T2` per crop (E3's T2 fit, never refitted) — CPU

In [ ]:
if not (OUT / position_shift.scores_json(position_shift.PRIOR_NAME)).exists():
    run([sys.executable, '-m', 'core.tools.position_shift', 'prior', '--crops', CROPS,
         '--data-dir', L_V2, '--reference', PRIOR_REF, '--out-dir', OUT], log=OUT / 'prior.log')
print(json.loads((OUT / position_shift.scores_json(position_shift.PRIOR_NAME)).read_text()))

## 6. Read-out (R1–R4, X7, X8) — CPU

In [ ]:
run([sys.executable, '-m', 'core.tools.position_shift', 'readout', '--crops', CROPS, '--scores-dir', OUT,
     '--reference-model', REF['A3ign'], '--reference-prior', PRIOR_REF, '--out-dir', OUT], log=OUT / 'readout.log')
print((OUT / position_shift.READOUT_MD).read_text())

## 7. Record — then stop

Paste `Thesis-V2/outputs/REPORTS/v22_df8/df8_readout.md` to Claude (and `df8_readout.json` if asked). The X8 sentence
is mechanical: nothing is re-scored with other settings after reading it.